In [ ]:
# Install required packages (if not already installed)
!pip install -q mujoco brax jax jaxlib tensorboardX

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create a directory for our logs and models
!mkdir -p /content/drive/MyDrive/Growbot/logs
!mkdir -p /content/drive/MyDrive/Growbot/models

# Load TensorBoard in the output of this cell
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/Growbot/logs

In [ ]:
import jax
from jax import numpy as jnp
import mujoco
from mujoco import mjx
from brax import envs
from brax.envs.base import PipelineEnv, State
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from brax.io import model
import functools

from torch.utils.tensorboard import SummaryWriter
import os
from datetime import datetime

# 1. Modified MJCF (Flat plane for baseline training)
XML_STRING = """
<mujoco model="Growbot">
  <option gravity="0 0 -9.81" timestep="0.005" integrator="RK4" solver="CG" iterations="10" ls_iterations="10"/>

  <default>
    <geom friction="1.2 0.1 0.1" solref="0.005 1" solimp="0.99 0.99 0.01" condim="3"/>
    <joint damping="0.03" armature="0.002"/>
    <position kp="0.75" kv="0.05" ctrlrange="-1.57 1.57" forcerange="-1.5 1.5"/>
  </default>

  <worldbody>
    <light name="sun" pos="0 0 3" dir="0 0 -1" diffuse="0.8 0.8 0.8" specular="0.2 0.2 0.2" castshadow="true"/>
    <geom name="floor" type="plane" size="0 0 0.05" rgba="0.76 0.87 0.70 1"/>

    <body name="base_body" pos="0 0 0.3">
      <joint name="root_joint" type="free"/>
      <geom name="torso_geom" type="box" size="0.0825 0.045 0.012" mass="0.3" rgba="0.7 0.7 0.7 1"/>

      <body name="right_leg" pos="0 -0.048 0">
        <joint name="joint_1" type="hinge" axis="0 1 0" limited="true" range="-90 90" />
        <geom name="lower_leg_1" type="box" size="0.0105 0.0065 0.0425" pos="0 0 -0.0425" mass="0.03" rgba="0.2 0.2 0.8 1"/>
      </body>

      <body name="left_leg" pos="0 0.048 0">
        <joint name="joint_2" type="hinge" axis="0 1 0" limited="true" range="-90 90" />
        <geom name="lower_leg_2" type="box" size="0.0105 0.0065 0.0425" pos="0 0 -0.0425" mass="0.03" rgba="0.2 0.2 0.8 1"/>
      </body>
    </body>
  </worldbody>

  <actuator>
    <position name="servo_1" joint="joint_1"/>
    <position name="servo_2" joint="joint_2"/>
  </actuator>
</mujoco>
"""



# Define the callback function that Brax will trigger
def save_checkpoint(current_step, make_policy, params):
    # Save the parameters with the current step count in the filename
    ckpt_path = f"{checkpoint_dir}/growbot_step_{current_step}.pkl"
    model.save_params(ckpt_path, params)

    # We use a simple print statement so you know it was successfully written to Drive
    print(f"  [➔ Checkpoint seamlessly saved to Drive at step {current_step}]")

# 2. Define the Brax Environment
class GrowbotEnv(PipelineEnv):
    def __init__(self, **kwargs):
        mj_model = mujoco.MjModel.from_xml_string(XML_STRING)
        mj_model.opt.solver = mujoco.mjtSolver.mjSOL_CG
        mj_model.opt.iterations = 6
        mj_model.opt.ls_iterations = 6
        sys = mjx.put_model(mj_model)
        super().__init__(sys=sys, backend='mjx', n_frames=4, **kwargs)

    # --- THE FIX ---
    @property
    def action_size(self):
        """Bypass Brax's default act_size() call since MJX models use .nu"""
        return self.sys.nu
    # ---------------

    def reset(self, rng: jax.Array) -> State:
        rng, rng1, rng2 = jax.random.split(rng, 3)
        qpos = self.sys.qpos0 + jax.random.uniform(rng1, (self.sys.nq,), minval=-0.05, maxval=0.05)
        qvel = jax.random.uniform(rng2, (self.sys.nv,), minval=-0.05, maxval=0.05)

        data = self.pipeline_init(qpos, qvel)

        # Initialize 5 frames of 2-DOF actions as zeros
        action_history = jnp.zeros((5, 2))

        # Split the RNG key: one to generate noise now, one to save for the next step
        rng, obs_rng = jax.random.split(rng)

        obs = self._get_obs(data, action_history, obs_rng)

        reward, done, zero = jnp.zeros(3)
        metrics = {
            'reward_forward': zero,
            'reward_survive': zero,
            'reward_ctrl_cost': zero,
            'x_position': zero
        }

        # NEW: Store our history and the next step's RNG key in the info dict
        info = {'action_history': action_history, 'rng': rng}

        # Make sure to return `info` as the 6th argument!
        return State(data, obs, reward, done, metrics, info)

    def step(self, state: State, action: jax.Array) -> State:
        data0 = state.pipeline_state
        data = self.pipeline_step(data0, action)

        x_velocity = (data.qpos[0] - data0.qpos[0]) / self.dt
        reward_forward = x_velocity * 5.0
        ctrl_cost = 0.02 * jnp.sum(jnp.square(action))
        reward = reward_forward - ctrl_cost
        done = jnp.float32(0)

        # Extract the old history and RNG key from the state info
        old_history = state.info['action_history']
        rng, obs_rng = jax.random.split(state.info['rng'])

        # Shift history: add the new action at the top, keep the 4 most recent old actions
        new_history = jnp.concatenate([jnp.expand_dims(action, 0), old_history[:-1]])

        # Pass the updated history and the noise key
        obs = self._get_obs(data, new_history, obs_rng)

        state.metrics.update(
            reward_forward=reward_forward,
            reward_survive=jnp.float32(0.0),
            reward_ctrl_cost=ctrl_cost,
            x_position=data.qpos[0]
        )

        # Copy the existing info dictionary so we don't delete Brax's hidden keys
        new_info = state.info.copy()

        # Update only our specific variables
        new_info['action_history'] = new_history
        new_info['rng'] = rng

        # Add info=new_info to the replacement call
        return state.replace(pipeline_state=data, obs=obs, reward=reward, done=done, info=new_info)

    def _get_obs(self, data: mjx.Data, action_history: jax.Array, rng: jax.Array) -> jax.Array:
        # 1. IMU: Extract Quaternion & convert to Euler
        w, x, y, z = data.qpos[3], data.qpos[4], data.qpos[5], data.qpos[6]
        roll = jnp.arctan2(2.0 * (w * x + y * z), 1.0 - 2.0 * (x * x + y * y))
        pitch = jnp.arcsin(jnp.clip(2.0 * (w * y - z * x), -1.0, 1.0))
        yaw = jnp.arctan2(2.0 * (w * z + x * y), 1.0 - 2.0 * (y * y + z * z))
        imu_angles = jnp.array([roll, pitch, yaw])

        # 2. Gyroscope: Angular velocity of the free joint
        gyro = data.qvel[3:6]

        # 3. Add Noise
        rng_angles, rng_gyro = jax.random.split(rng)
        # Tweak these multipliers (0.05 and 0.1) based on your real IMU's datasheet variance
        noisy_angles = imu_angles + jax.random.normal(rng_angles, (3,)) * 0.05
        noisy_gyro = gyro + jax.random.normal(rng_gyro, (3,)) * 0.1

        # 4. Action History: Flatten the 5x2 matrix into a 1D array of 10 elements
        flat_history = action_history.flatten()

        # Final observation size: 3 (angles) + 3 (gyro) + 10 (history) = 16 elements
        return jnp.concatenate([noisy_angles, noisy_gyro, flat_history])

# Register Environment
envs.register_environment('growbot', GrowbotEnv)

if __name__ == "__main__":
    # Setup TensorBoard Writer
    # checkpoint_path = "/content/drive/MyDrive/Growbot/models/growbot_ppo_prev_motor_output_obs20260611-175847.pkl"
    # print(f"Loading pre-trained weights from: {checkpoint_path}")
    # loaded_params = model.load_params(checkpoint_path)
    run_desc = '2nd_try_obs_stack_gyro_data_phone_body_85mm'
    run_name = run_desc + datetime.now().strftime('%Y%m%d-%H%M%S')
    # Create a dedicated checkpoints folder inside your Google Drive
    checkpoint_dir = f"/content/drive/MyDrive/Growbot/models/checkpoints_{run_name}"
    os.makedirs(checkpoint_dir, exist_ok=True)
    log_dir = f"/content/drive/MyDrive/Growbot/logs/{run_name}"
    writer = SummaryWriter(log_dir)
    print(f"Logging TensorBoard data to: {log_dir}")

    env = envs.get_environment('growbot')

    network_factory = functools.partial(
            ppo_networks.make_ppo_networks,
            policy_hidden_layer_sizes=(64, 64),
            value_hidden_layer_sizes=(64, 64)
        )

    # Custom Progress function to push data to TensorBoard
    def progress(num_steps, metrics):
        # Brax automatically prefixes environment metrics with 'eval/episode_'
        # We extract them and log them categorically to TensorBoard
        writer.add_scalar('Reward/1_Total', metrics['eval/episode_reward'], num_steps)
        writer.add_scalar('Reward/2_Forward', metrics['eval/episode_reward_forward'], num_steps)
        writer.add_scalar('Reward/3_Survive', metrics['eval/episode_reward_survive'], num_steps)
        writer.add_scalar('Penalty/Control_Cost', metrics['eval/episode_reward_ctrl_cost'], num_steps)
        writer.add_scalar('Metrics/X_Position', metrics['eval/episode_x_position'], num_steps)

        print(f"Step: {num_steps} | Tot: {metrics['eval/episode_reward']:.1f} | Fwd: {metrics['eval/episode_reward_forward']:.1f} | Ctrl: {metrics['eval/episode_reward_ctrl_cost']:.1f}")

    print("Starting PPO training...")

    make_inference_fn, params, _ = ppo.train(
        environment=env,
        num_timesteps=60_000_000,    # Increased to 15M for a longer training run
        num_evals=90,
        reward_scaling=1.0,
        episode_length=1000,
        normalize_observations=True,
        action_repeat=1,
        unroll_length=20,
        num_minibatches=32,
        num_updates_per_batch=4,
        discounting=0.99,
        learning_rate=3e-4,
        entropy_cost=0.05,           # Increased entropy to force more random exploration
        num_envs=2048,
        batch_size=1024,
        network_factory=network_factory,
        progress_fn=progress,
        policy_params_fn=save_checkpoint,
        # restore_params=loaded_params,
        seed=42
    )

    writer.close()

    # Save final weights to Google Drive
    save_path = f"/content/drive/MyDrive/Growbot/models/growbot_ppo_{run_name}.pkl"
    model.save_params(save_path, params)
    print(f"\nTraining complete! Model saved securely to: {save_path}")